In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import time
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
# macbook : "mps"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
import os

with open("mov.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Total characters: {len(text):,}")
print(f"First 200 characters:\n{text[:200]}")

Total characters: 130,853
First 200 characters:
SCENE:
Venice, and PORTIA'S house at Belmont


ACT 1. SCENE I.
Venice. A street

Enter ANTONIO, SALERIO, and SOLANIO

  ANTONIO. In sooth, I know not why I am so sad.
    It wearies me; you say it wea


In [3]:
# built character vocubulary
chars=sorted(set(text))
vocab_size=len(chars)

# charater to index and vice versa
char_to_idx={c:i for i,c in enumerate(chars)}
idx_to_char={i:c for i,c in enumerate(chars)}

#encode and decode function for inputs
def encode(s):
    return[char_to_idx[c] for c in s]
def decode(ids):
    return "".join([idx_to_char[i] for i in ids])

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")
print(f"\nExample encoding:")
print(f"  'Antonio' -> {encode('Antonio')}")
print(f"  {encode('Antonio')} -> '{decode(encode('Antonio'))}'")

Vocabulary size: 78
Characters: 
 !'(),-.0123456789:;<>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyz

Example encoding:
  'Antonio' -> [24, 65, 71, 66, 65, 60, 66]
  [24, 65, 71, 66, 65, 60, 66] -> 'Antonio'


In [4]:
#split the data into two parts, 85% for model learning and 15% for model validaion
data=torch.tensor(encode(text), dtype=torch.long)
n=int(0.85*len(data))
train_data=data[:n]
val_data=data[n:]
print(f"Training token: {len(train_data):,}")
print(f"Validation token: {len(val_data):,}")

Training token: 111,225
Validation token: 19,628


In [5]:
# Batching: grab random chunks of text
def get_batch(split, batch_size, context_length):
    d=train_data if split=="train" else val_data
    ix=torch.randint(len(d)-context_length, (batch_size,))
    x=torch.stack([d[i:i+context_length] for i in ix])
    y=torch.stack([d[i+1:i+context_length+1] for i in ix])
    return x.to(device),y.to(device)

# Quick test of batching
xb, yb = get_batch("train", batch_size=4, context_length=15)
print(f"Input shape:  {xb.shape}  (batch_size x context_length)")
print(f"Target shape: {yb.shape}")
print(f"\nExample (first sequence):")
print(f"  Input:  {decode(xb[0].tolist())!r}")
print(f"  Target: {decode(yb[0].tolist())!r}")
print(f"  (Target is input shifted by 1 character)")

Input shape:  torch.Size([4, 15])  (batch_size x context_length)
Target shape: torch.Size([4, 15])

Example (first sequence):
  Input:  'er. I\n    am fa'
  Target: 'r. I\n    am fam'
  (Target is input shifted by 1 character)


# **RMSNorm**
RMSNorm skips the mean subtraction and the shift as of in layerNorm. Just divide by the root-mean-square, then scale. Simpler, faster, works just as well.

**Root Mean Square Layer Normalization.**

Simpler than LayerNorm:
- No mean subtraction
- No bias/shift parameter
- Just: x / RMS(x) * learnable_scale

**DROPOUT**
- We will also use Dropout throughout the model.
- Dropout randomly zeroes some values during training,
- forcing the model to not rely on any single feature.
- This prevents memorization (overfitting).

In [6]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps=eps
        self.weight=nn.Parameter(torch.ones(dim))

    def forward(self,x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight
DROPOUT=0.2

# --- Demo ---
demo_x = 1000 + (2000 - 1000) * torch.rand(2, 4, 10)
norm = RMSNorm(10)
demo_out = norm(demo_x)

print("RMSNorm demo:")
print(f"  Input  - mean: {demo_x.mean():.3f}, std: {demo_x.std():.3f}")
print(f"  Output - mean: {demo_out.mean():.3f}, std: {demo_out.std():.3f}")
print(f"  Input range:  [{demo_x.min():.3f}, {demo_x.max():.3f}]")
print(f"  Output range: [{demo_out.min():.3f}, {demo_out.max():.3f}]")
print(f"\n  Parameters: just a scale vector of size {norm.weight.shape}")

RMSNorm demo:
  Input  - mean: 1494.017, std: 298.374
  Output - mean: 0.982, std: 0.189
  Input range:  [1002.674, 1979.582]
  Output range: [0.614, 1.351]

  Parameters: just a scale vector of size torch.Size([10])


# **RoPE**
RoPE rotates Q and K vectors based on position. The dot product then naturally captures relative distance between tokens.

Like a clock: the angle between 3 o'clock and 5 o'clock = the angle between 7 o'clock and 9 o'clock. Absolute position doesn't matter, only the gap.

In [8]:
#precompute give different frequencies to each pair and produce angle according to them 
def precompute_rope_freqs(head_dim, max_seq_len, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(max_seq_len).float()
    angles = torch.outer(positions, freqs)
    return torch.cos(angles), torch.sin(angles)

#applying RoPE
def apply_rope(x, cos, sin):
    seq_len = x.shape[2]
    cos = cos[:seq_len].unsqueeze(0).unsqueeze(0)  # [1, 1, seq, hd//2]
    sin = sin[:seq_len].unsqueeze(0).unsqueeze(0)
    x1 = x[..., ::2]   # even dims
    x2 = x[..., 1::2]  # odd dims
    out1 = x1 * cos - x2 * sin
    out2 = x1 * sin + x2 * cos
    return torch.stack([out1, out2], dim=-1).flatten(-2)

# **GQA**
GQA keeps many Query heads but makes groups of Query heads share fewer Key/Value heads, while every K/V head still contains information across all tokens.reducing K/V heads doesn't mean reducing the number of tokens or automatically making the model unable to find the right answer.

In [ ]:
#repear_kv is used to copy key,value. like if we have K1 if n_rep=3 it will become K1 K1 K1
def repeat_kv(x, n_rep):
    if(n_rep==1):
        return x
    b,n_kv,seq,hd=x.shape
    return(x[:,:,None,:,:].expand(b,n_kv,n_rep,seq,hd).reshape(b,n_kv*n_rep,seq,hd))

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert d_model%n_heads==0
        assert n_heads%n_kv_heads==0

        #we use self. because we want that value in further program
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.n_rep=n_heads//n_kv_heads
        self.head_dim=d_model//n_heads

        #creating lyers for query,key,value,output
        self.q_proj=nn.Linear(d_model, n_heads*self.head_dim, bias=False)
        self.k_proj=nn.Linear(d_model, n_kv_heads*self.head_dim, bias=False)
        self.v_proj=nn.Linear(d_model, n_kv_heads*self.head_dim, bias=False)
        self.o_proj=nn.Linear(n_heads*self.head_dim, d_model, bias=False)

        def forward(self, x, rope_cos, rope_sin):
            b, seq, _ = x.shape

            #project q,k,v
            q=self.q_proj(x)
            k=self.k_proj(x)
            v=self.k_proj(x)